# Gateway tracking with an uncertain halo observer

Compare Gateway orbit estimation over one observer period using (1) exact observer knowledge and (2) an observer orbit estimated from lunar surface observations. Both cases use the same truth trajectories, Gateway prior and noisy Gateway measurements.

In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "configs/baseline.json").exists() and (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from within the project checkout")
sys.path.insert(0, str(ROOT / "src"))
from halo_tracking.study import load_config
cfg = load_config(ROOT / "configs/baseline.json")
cfg["simulation"]["periods"] = 1.0  # One nominal observer period


## 1. Matched comparison

| Assumption | Perfect observer | LSO-estimated observer |
|---|---|---|
| Observer state supplied to Gateway estimator | Exact true state | State estimated from LSO angles |
| Initial observer uncertainty | Zero | 1 km and 0.01 m/s per axis, 1σ |
| Initial Gateway uncertainty | 1 km and 0.01 m/s per axis, 1σ | Same |
| Observer-to-Gateway camera | 1 arcsecond per angle | Same measurements and noise |
| LSO | Not needed | 30° N at 270° E and 90° E; 85° S at 90° E; 1 arcsecond per angle |

These are the baseline settings; the configuration table below displays the values actually used. Both optical links use **angles only** in this focused comparison. The cadence is 30 minutes. Earth/Moon occultation is included for Gateway observations; the LSO uses the configured elevation mask. Optical brightness, Sun exclusions and camera attitude errors are not modeled.

The true observer remains on the supplied initial-value trajectory in both cases. The imperfect case introduces uncertainty in our **knowledge** of it, not a different physical orbit.

In [ ]:
from IPython.display import HTML, Markdown, display
settings = [
    ("Duration", f"1 observer nominal period = {cfg['orbit']['period_tu'] * cfg['time_unit_s'] / 86400:.2f} days"),
    ("Observer initial position / velocity 1σ", f"{cfg['estimation']['initial_position_sigma_km']:g} km / {1000*cfg['estimation']['initial_velocity_sigma_km_s']:g} m/s per axis"),
    ("Gateway initial position / velocity 1σ", f"{cfg['comparison']['gateway_position_sigma_km']:g} km / {1000*cfg['comparison']['gateway_velocity_sigma_km_s']:g} m/s per axis"),
    ("LSO angular noise 1σ", f"{cfg['measurement']['angle_sigma_arcsec']:g} arcsec per angle"),
    ("Gateway camera angular noise 1σ", f"{cfg['comparison']['gateway_camera_sigma_arcsec']:g} arcsec per angle")]
display(HTML("<table>"+"".join(f"<tr><th>{name}</th><td>{value}</td></tr>" for name,value in settings)+"</table>"))

In [ ]:
from halo_tracking.gateway_comparison import run_gateway_comparison, save_comparison
result = run_gateway_comparison(cfg)
report = save_comparison(result, ROOT / "results/gateway_comparison")

## 2. Geometry and what the LSO knows

The first figure spans exactly one observer nominal period (Gateway repeats at its own period). It is context: both true trajectories are identical across the two estimation cases. The dots mark their initial states. The axis scale is in thousands of kilometers, with equal physical scale on all three axes.

The second figure shows the observer's estimated-state error and its local covariance prediction. The shaded intervals contain observations from at least one LSO. The three-site network is evaluated at every scheduled epoch. **RSS σ is a covariance summary, not a guaranteed bound on the error magnitude.**

In [ ]:
from halo_tracking.plotting import geometry, observer_knowledge, gateway_comparison_figures, state_histories, export
fig_geometry = geometry(result)
export(fig_geometry, ROOT / "results/gateway_comparison/geometry.png")

In [ ]:
fig_observer = observer_knowledge(result)
export(fig_observer, ROOT / "results/gateway_comparison/observer.png")

## Observer position, velocity and acceleration

The solid curve is the perfect observer state (truth); the dashed curve is the LSO-predicted state. Components use Moon-centered inertial axes. These curves nearly overlap at orbit scale; the error plot above resolves their differences.

In [ ]:
fig_observer_states = state_histories(result, 'observer')
export(fig_observer_states, ROOT / 'results/gateway_comparison/observer_states.png')

## Gateway position, velocity and acceleration

Compare the true Gateway trajectory with the two tracked trajectories: one uses the perfect observer and one uses the LSO-predicted observer. Their small separation is quantified by the error plot and RMSE table below.

In [ ]:
fig_gateway_states = state_histories(result, 'gateway')
export(fig_gateway_states, ROOT / 'results/gateway_comparison/gateway_states.png')

## 3. What changes in Gateway tracking

**Actual-error plot:** distance between the estimated and true Gateway state in this one shared noise realization. Lower is better. It is an actual orbit estimate, not just a pointing-error projection.

**Predicted-uncertainty plot:** square root of the position/velocity covariance-block trace. Lower means the estimator predicts better knowledge. These are local linearized predictions; they can be optimistic if the approximation is inaccurate.

In the imperfect case, the estimator carries observer–Gateway cross-covariance through time. Gateway measurements do not refine the observer: its state knowledge comes from the LSO. LSO updates can also correct Gateway indirectly through the retained correlation.

In [ ]:
figures = gateway_comparison_figures(result)
fig_errors = figures["errors"]
export(fig_errors, ROOT / "results/gateway_comparison/errors.png")
import matplotlib.pyplot as plt
plt.close(figures["uncertainty"])

In [ ]:
figures_uncertainty = gateway_comparison_figures(result)
plt.close(figures_uncertainty["errors"])
fig_uncertainty = figures_uncertainty["uncertainty"]
export(fig_uncertainty, ROOT / "results/gateway_comparison/uncertainty.png")

## Model and limitations

Propagation uses the Earth–Moon CR3BP. The plots transform normalized rotating states into Moon-centered inertial position, velocity and acceleration. Acceleration is calculated from each state using the dynamics; it is not independently estimated.

The imperfect case retains observer–Gateway cross-covariance. LSO measurements update observer knowledge; Gateway measurements update Gateway without refining the observer. Initial covariances are diagonal, with zero observer–Gateway cross-covariance. The forecast uses zero innovations along the nominal trajectories; the actual estimate uses noisy measurements.

The supplied `SNRHL2` state is a southern NRHO initial condition rounded to four decimals. Its one-period closure miss is about 99 km. This study assumes no stationkeeping or unmodeled forces. Local filter covariance can be overconfident near nonlinear updates, so predicted uncertainty is a conditional comparison rather than a validated error bound.

Gateway's initial state comes from [the earlier cislunar project](https://github.com/scottblender/Gradient-Free-Optimization-for-Cislunar-SDA-2/blob/main/scripts/build_target_case_database.m). Both phases start at model time zero, without a mission epoch. Both trajectories use LU = 384400 km and TU = 375190.25852 s; the source Gateway project used TU = 375695 s.

## Uncontrolled observer departure from a corrected periodic reference

The physical observer truth starts at the original four-decimal SNRHL2 catalog state and is propagated continuously without station-keeping. **Only the nominal reference** is differentially corrected to a nearby periodic CR3BP orbit. All 20 horizons use that corrected nominal period, \(T\), so the observer's starting state and the reference are intentionally not identical.

The catalog's rounded state would miss closure by roughly 99 km after its supplied period, but the corrected nominal reference closes to a much tighter numerical tolerance. The first-period RMS position error is therefore **nonzero**, reflecting the initial difference between truth and nominal; this is expected, not a failed solver or a reference reset.

Position departure means distance between the continuous observer truth and its equal-phase periodic nominal position. It is **not** distance to Gateway, and a large departure is not by itself a validated station-keeping deadline. The ensemble shows sensitivity to additional seeded 1-km position and 0.01-m/s velocity perturbations.

In [ ]:
from halo_tracking.departure import reference_departure, save_departure
from halo_tracking.plotting import departure_figures
departure = reference_departure(cfg, periods=20, ensemble_count=8)
save_departure(departure, cfg, ROOT / 'results/departure')
departure_plots = departure_figures(departure, cfg)
export(departure_plots['departure'], ROOT / 'results/departure/departure.png')
export(departure_plots['period_growth'], ROOT / 'results/departure/period_growth.png')

In [ ]:
closure = departure['diagnostics']
periods = [2, 4, 6, 8, 10, 20]
rows = ''.join(f"<tr><td>{k}</td><td>{departure['period_position_rms_km'][k-1]:,.1f}</td>"
               f"<td>{departure['period_velocity_rms_m_s'][k-1]:,.3f}</td></tr>" for k in periods)
display(HTML('<table><tr><th>Period</th><th>RMS position departure (km)</th>'
             '<th>RMS velocity departure (m/s)</th></tr>' + rows + '</table>'))
display(Markdown(f"""
The corrected **nominal** one-period closure is **{closure['position_closure_km']:.6f} km** and **{closure['velocity_closure_m_s']:.6f} m/s**. The original rounded **catalog** state's one-period closure error is **{closure['catalog_closure_km']:.1f} km**. The initial rounded truth-to-nominal position offset is **{closure['initial_nominal_offset_km']:.1f} km**.

Rapid divergence is evaluated from this particular rounded-reference simulation and the 1 km / 0.01 m/s initial perturbation ensemble. Do not interpret transient decreases as stabilization, or set an operational cutoff before defining position/velocity tolerances.

The next stationkeeping decision is an acceptable departure corridor and prediction horizon. These curves support choosing that requirement; they do not establish the latest safe maneuver time by themselves.
"""))

## Cumulative uncontrolled observer trajectories

All grid panels show increasingly long prefixes of **the same uninterrupted propagation from the original rounded catalog state**: \(0\to T\), \(0\to2T\), ..., \(0\to20T\), where \(T\) is the corrected nominal halo period.

The dashed gray reference is a *differentially corrected periodic nominal trajectory*, repeated at each phase. **It is not the spacecraft's truth trajectory.** The first panel can therefore already show an offset between the truth and the reference. Each panel reports the maximum nominal-reference departure reached *any time up to that horizon*.

The first grid auto-scales each panel to show the entire growing trajectory, and a second grid holds the corrected nominal reference axis limits fixed. All panels have the same initial truth state; no restarts or impulses are applied.

In [ ]:
from halo_tracking.plotting import period_trajectory_grid
fig_period_grid = period_trajectory_grid(departure, cfg, scale_mode='local')
export(fig_period_grid, ROOT / 'results/departure/observer_period_grid.png')
display(fig_period_grid)
plt.close(fig_period_grid)

fig_reference_grid = period_trajectory_grid(departure, cfg, scale_mode='reference')
export(fig_reference_grid, ROOT / 'results/departure/observer_period_grid_fixed_scale.png')
plt.close(fig_reference_grid)

## How long can Gateway be predicted without new observations?

This is a **blind-prediction sensitivity experiment**, replacing the redundant cumulative Gateway RMSE table. The initial one-period tracking figures remain as baseline context. It keeps the original freely propagating halo observer, the physical Sun/Earth keep-outs, and the continuously evolving EKF states and covariances.

**The controlled difference:** Gateway's simulated truth receives an illustrative fixed-direction inertial acceleration (default **2 × 10⁻⁹ m/s²**); both estimators still predict **unforced CR3BP** and do not know this acceleration. To isolate what happens when Gateway observations disappear, **no Gateway measurements are accepted after 8 nominal observer periods** (one period for the 2-period CI run). Lunar-surface measurements retain their normal geometric constraints. This deliberate blackout is a controlled experiment, **not a claim that the actual optics always lose visibility exactly at 8T**.

The figure separates **instantaneous position error** (actual error now), **rolling 24-hour RMSE** (recent error rather than an entire mission average), **filter-reported local uncertainty**, and **days since the last accepted Gateway observation**. Forecast uncertainty is conditional on the incorrect model; the difference between error and uncertainty is itself a useful consistency diagnostic.

The two tables below show horizon-by-horizon prediction degradation and, for each specified position-error tolerance, the first crossing after the blackout. A tolerance not exceeded before the simulation ends is **right-censored**—not proof of indefinite accuracy. Optical range caps and disturbance magnitude are exploratory assumptions, not calibrated Gateway performance limits.

In [ ]:
import numpy as np
import os
observer_period_count = int(os.environ.get('HALO_GATEWAY_PERIODS', '20'))

def render_metrics_table(title, headers, records):
    """Compact, horizontally scrollable table for blind prediction results."""
    headings = ''.join(
        '<th style="padding:8px 11px;text-align:right;'
        'background:#eef3f8;border-bottom:1px solid #cbd5e1;">'
        + str(label) + '</th>' for label in headers)
    rows = ''.join(
        '<tr style="background:' + ('#f6f8fc' if i%2 else 'white') + '">'
        + ''.join('<td style="text-align:right;padding:7px 11px;'
                  'border-bottom:1px solid #e5e9f0;white-space:nowrap;">'
                  + str(value) + '</td>' for value in record)
        + '</tr>' for i, record in enumerate(records))
    return (
        '<div style="margin:16px 0 6px;font-weight:600;">' + title +
        '</div><div style="overflow-x:auto;max-height:590px;'
        'border:1px solid #d5dde7;border-radius:8px;">'
        '<table style="border-collapse:collapse;min-width:800px;'
        'font:12px system-ui,sans-serif;font-variant-numeric:tabular-nums;">'
        '<thead><tr>' + headings + '</tr></thead><tbody>' + rows +
        '</tbody></table></div>')

from halo_tracking.gateway_prediction import (
    run_gateway_prediction_study, save_gateway_prediction_study,
    load_gateway_prediction_study
)
from halo_tracking.plotting import gateway_prediction_figure

prediction_periods = observer_period_count
blackout_period = min(int(os.environ.get('HALO_GATEWAY_BLACKOUT_PERIODS', '8')),
                      prediction_periods - 1)
truth_acceleration = float(os.environ.get(
    'HALO_GATEWAY_TRUTH_ACCELERATION_M_S2', '2e-9'))
prediction_output = ROOT / 'results/gateway_prediction'
prediction_kwargs = dict(
    periods=prediction_periods, blackout_period=blackout_period,
    disturbance_m_s2=truth_acceleration, rolling_window_days=1.,
    thresholds_km=(1., 10., 100.))
prediction = load_gateway_prediction_study(
    cfg, prediction_output, **prediction_kwargs)
if prediction is None:
    prediction = run_gateway_prediction_study(cfg, **prediction_kwargs)
    save_gateway_prediction_study(prediction, prediction_output)
    print('Computed new truth-mismatch / forced-blackout prediction scenario.')
else:
    print('Loaded matching saved prediction scenario and time histories.')

fig = gateway_prediction_figure(prediction)
export(fig, prediction_output / 'blind_prediction_endurance.png')
display(fig)
plt.close(fig)

summary = prediction['summary']
display(Markdown(
    f"**Model mismatch:** {summary['disturbance_m_s2']:.2e} m/s² fixed "
    f"inertial direction; **forced blackout starts:** "
    f"{summary['blackout_elapsed_days']:.2f} days; "
    f"**last accepted Gateway observation:** "
    f"{summary['last_accepted_gateway_observation_days']} days. "
    "The estimator uses unforced CR3BP throughout."
))

cols = ['Horizon', 'New Gateway obs.', 'Days since last obs.',
        'Exact obs. error (km)', 'LSO obs. error (km)',
        'Exact obs. 24h RMSE (km)', 'LSO obs. 24h RMSE (km)']
hrows = []
for k in range(1, prediction_periods + 1):
    pair = {r['case']: r for r in prediction['rows']
            if r['horizon_periods'] == k}
    a, b = pair['perfect'], pair['lso_estimated']
    age = a['days_since_gateway_observation']
    hrows.append([f'0 → {k}T', a['gateway_observations_latest_period'],
                  f'{age:.2f}' if np.isfinite(age) else 'No observation yet',
                  f"{a['instantaneous_position_error_km']:.3f}",
                  f"{b['instantaneous_position_error_km']:.3f}",
                  f"{a['rolling_position_rmse_km']:.3f}",
                  f"{b['rolling_position_rmse_km']:.3f}"])
display(HTML(render_metrics_table(
    'Gateway error at each cumulative-horizon endpoint', cols, hrows)))

limits_rows = []
for r in prediction['threshold_rows']:
    value = r['crossing_days_since_last_observation']
    when = (f'{value:.2f}' if value is not None
            else 'Not reached by study end')
    limits_rows.append([r['case'],
                        f"{r['position_threshold_km']:g}", when])
display(HTML(render_metrics_table(
    'First time position-error tolerance is exceeded after blackout',
    ['Observer knowledge', 'Error tolerance (km)',
     'Days after last accepted Gateway observation'], limits_rows)))

## Station-keeping system identification — step 1: generate guidance examples

**Reformulated prediction task:** estimate the **three-component corrective impulse \(\Delta\mathbf v\)** required to keep the observer near its differentially corrected periodic halo reference. This dataset is *not* for next-state prediction. The nominal period and position/velocity reference come from differential correction; physical simulated observer trajectories begin either near the rounded catalog seed or near the corrected nominal state.

**Exploratory maneuver policy**, chosen using the uncontrolled multi-period propagation:

- **Maneuver trigger:** \(\|\Delta\mathbf r\| \ge 50\) km **or** \(\|\Delta\mathbf v_\mathrm{state}\| \ge 0.5\) m/s.
- **Study corridor:** 100 km position and 1 m/s velocity departure from the nominal reference.
- **Single-burn guidance:** optimize an impulsive velocity change against the nominal state **0.25T** later, with a 10 m/s maximum allowed burn (illustrative upper bound, not a spacecraft specification).

The rounded trajectory begins about **23 km** and **0.13 m/s** from the corrected nominal state and reaches about **110 km** and **0.9 m/s** around **0.5T**. Thus the selected triggers act *before* the rapidly growing departures seen over multiple periods. These thresholds are study assumptions to refine with a station-keeping trade analysis, not certified Gateway/observer requirements.

Each independent **2-period** trajectory is sampled for optical observations and estimated with a multi-LSO EKF; correction decisions are taken four times per nominal period. The feature set includes **estimated** rotating state, nominal state, reference-relative position/velocity, time/phase, EKF uncertainty, measurement availability, and time since the last LSO observation. Teacher targets are local CR3BP STM-based, regularized, *velocity-only* corrections. Every proposed burn is checked with nonlinear forward propagation; burns that exceed the budget or fail to reduce predicted error are **masked** instead of silently clipped. No-burn decisions have zero targets.

Simulation **truth states are saved for evaluation only, not as deployable training inputs**. The generated labels depend on noisy navigation estimates; no controller or ML model is trained in this step. Training, validation and test splits contain independent whole trajectories.

In [ ]:
from halo_tracking.identification import generate_dataset
identification_summary = generate_dataset(
    cfg, ROOT / 'results/system_identification')
headers = ['Split', 'Arcs', 'Decisions', 'Triggered',
           'Feasible burns', 'Rejected burns', 'No-burn samples', 'LSO visibility']
rows = ''.join(
    '<tr>' + ''.join(f'<td>{value}</td>' for value in
    [name, item['arcs'], item['decisions'], item['triggered'],
     item['valid_triggered'], item['rejected'], item['no_burn'],
     f"{100*item['visible_fraction']:.1f}%"]) + '</tr>'
    for name, item in identification_summary.items())
display(HTML(
    '<table><tr>' + ''.join(f'<th>{h}</th>' for h in headers) +
    '</tr>' + rows + '</table>'))
display(Markdown(
    'Saved **training.npz**, **validation.npz**, **test.npz** and '
    '**metadata.json** to `results/system_identification/`. '
    'Training targets: **target_delta_v_rotating_m_s**. '
    'Always mask examples with **maneuver_label_valid=False**.'
))